In [1]:
import requests
import pandas as pd
import os
from pathlib import Path

In [3]:

# ============================================================
# 1. STRATEGIC COORDINATE ANCHORS - MUMBAI WESTERN ZONES
# ============================================================

FLOOD_ZONES = [
    {
        "zone_id": 0,
        "zone_name": "Bandra-Santacruz",
        "basin": "Mithi River Basin",
        "lat": 19.0600,
        "lon": 72.8500
    },
    {
        "zone_id": 1,
        "zone_name": "Andheri-VileParle",
        "basin": "Oshiwara Basin",
        "lat": 19.1200,
        "lon": 72.8400
    },
    {
        "zone_id": 2,
        "zone_name": "Goregaon-Malad",
        "basin": "Poisar Basin",
        "lat": 19.1700,
        "lon": 72.8500
    },
    {
        "zone_id": 3,
        "zone_name": "Kandivali-Borivali",
        "basin": "Dahisar River Basin",
        "lat": 19.2300,
        "lon": 72.8600
    }
]


# ============================================================
# 2. OUTPUT DIRECTORY - JUPYTER NOTEBOOK
# ============================================================

OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset storage location:", OUTPUT_DIR)


# ============================================================
# 3. FETCH FLOOD DISCHARGE DATA FROM OPEN-METEO GloFAS
# ============================================================

def fetch_flood_discharge(
    start_date="2023-01-01",
    end_date="2024-12-31"
):

    print("\nFetching GloFAS hydrological river discharge...")
    print("Source: Open-Meteo Flood API")

    latitudes = ",".join(
        str(zone["lat"]) for zone in FLOOD_ZONES
    )

    longitudes = ",".join(
        str(zone["lon"]) for zone in FLOOD_ZONES
    )

    api_url = "https://flood-api.open-meteo.com/v1/flood"

    params = {
        "latitude": latitudes,
        "longitude": longitudes,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "river_discharge",
        "models": "seamless_v4"
    }

    try:

        response = requests.get(
            api_url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        # Handle single or multiple location responses
        if isinstance(data, dict):
            data = [data]

        all_zone_dfs = []

        # ====================================================
        # 4. PROCESS EACH FLOOD ZONE
        # ====================================================

        for idx, location in enumerate(data):

            if idx >= len(FLOOD_ZONES):
                break

            meta = FLOOD_ZONES[idx]

            daily = location.get("daily", {})

            dates = daily.get("time", [])
            discharge = daily.get("river_discharge", [])

            if not dates:
                print(
                    f"No discharge data for {meta['zone_name']}"
                )
                continue

            df_zone = pd.DataFrame({

                "date": pd.to_datetime(dates),

                "zone_id": meta["zone_id"],

                "zone_name": meta["zone_name"],

                "basin_name": meta["basin"],

                "river_discharge_m3s": discharge

            })

            all_zone_dfs.append(df_zone)

            print(
                f"Processed {meta['zone_name']}: "
                f"{len(df_zone)} records"
            )

        if not all_zone_dfs:
            print("No flood discharge data received.")
            return None

        # ====================================================
        # 5. COMBINE ALL ZONE DATASETS
        # ====================================================

        df_flood = pd.concat(
            all_zone_dfs,
            ignore_index=True
        )

        # Sort records chronologically within each zone
        df_flood = df_flood.sort_values(
            ["zone_name", "date"]
        ).reset_index(drop=True)

        # ====================================================
        # 6. FEATURE ENGINEERING
        # ====================================================

        # Calculate 3-day rolling discharge average
        df_flood["discharge_3d_rolling_avg"] = (

            df_flood.groupby("zone_name")[
                "river_discharge_m3s"
            ]

            .transform(
                lambda s: s.rolling(
                    window=3,
                    min_periods=1
                ).mean()
            )
        )

        # ====================================================
        # 7. SAVE DATASETS
        # ====================================================

        full_output = (
            OUTPUT_DIR / "mumbai_flood_discharge.csv"
        )

        sample_output = (
            OUTPUT_DIR / "flood_sample.csv"
        )

        df_flood.to_csv(
            full_output,
            index=False
        )

        df_flood.head(10).to_csv(
            sample_output,
            index=False
        )

        # ====================================================
        # 8. DATASET SUMMARY
        # ====================================================

        print("\n" + "=" * 60)
        print("FLOOD DISCHARGE DATA EXTRACTION COMPLETED")
        print("=" * 60)

        print("\nDataset shape:", df_flood.shape)

        print(
            "\nTotal records:",
            len(df_flood)
        )

        print(
            "Total zones:",
            df_flood["zone_name"].nunique()
        )

        print(
            "Date range:",
            df_flood["date"].min(),
            "to",
            df_flood["date"].max()
        )

        print("\nMissing values:")
        print(df_flood.isnull().sum())

        print("\nSaved files:")
        print("1.", full_output)
        print("2.", sample_output)

        print("\nFirst 5 records:")
        print(df_flood.head())

        return df_flood

    except requests.exceptions.RequestException as e:

        print("\nAPI request failed:", e)
        return None

    except Exception as e:

        print("\nProcessing error:", e)
        raise


# ============================================================
# 9. EXECUTE DATA EXTRACTION
# ============================================================

df_flood = fetch_flood_discharge(
    start_date="2023-01-01",
    end_date="2024-12-31"
)


Dataset storage location: C:\Users\sahil\OneDrive\Desktop\Documents\Data Scientist SS\Smart-Urban-Operations-Platform\backend\scripts

Fetching GloFAS hydrological river discharge...
Source: Open-Meteo Flood API
Processed Bandra-Santacruz: 731 records
Processed Andheri-VileParle: 731 records
Processed Goregaon-Malad: 731 records
Processed Kandivali-Borivali: 731 records

FLOOD DISCHARGE DATA EXTRACTION COMPLETED

Dataset shape: (2924, 6)

Total records: 2924
Total zones: 4
Date range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00

Missing values:
date                        0
zone_id                     0
zone_name                   0
basin_name                  0
river_discharge_m3s         0
discharge_3d_rolling_avg    0
dtype: int64

Saved files:
1. C:\Users\sahil\OneDrive\Desktop\Documents\Data Scientist SS\Smart-Urban-Operations-Platform\backend\scripts\mumbai_flood_discharge.csv
2. C:\Users\sahil\OneDrive\Desktop\Documents\Data Scientist SS\Smart-Urban-Operations-Platform\backend\sc